# 03 選択フォルダ内データ一括処理

選んだメインフォルダの中にある**すべての末端フォルダ**（サブフォルダを持たないフォルダ）から加工対象ファイルを探し、
`01_最高温度関連グラフ作成.ipynb` と同じ処理を1ファイルずつまとめて行います。

## 使い方
1. セル1 → セル2 → セル3 の順に実行します。
2. セル2を実行すると「メインフォルダを選択してください」というダイアログが出るので、メインフォルダを選びます。
   - ダイアログが見当たらないときは、VS Code の後ろやタスクバーを確認してください。
3. セル2の実行結果に表示される「加工対象ファイル」の一覧が正しいか確認します。
4. セル3を実行します。最後に表示される「完了 / スキップ / エラー」の件数を確認してください。

## 出力
加工対象ファイルと**同じフォルダ**に、次の2つが作られます（01を単体で実行したときと同じです）。

| ファイル | 内容 |
|---|---|
| `グラフ出力結果_{元のファイル名}.xlsx` | 条件ごとの平均±標準偏差のグラフ（外れ値は×で表示） |
| `外れ値除外_{元のファイル名}.csv` | 外れ値のセルだけを空欄にしたCSV |

同じ名前のファイルがすでにある場合は**上書き**されます。

## 注意点
- **01と同じフォルダに置いてください。** このノートブックは `01_最高温度関連グラフ作成.ipynb` を読み込んで実行します。01を修正すると、一括処理にもそのまま反映されます。
- **加工対象**は、末端フォルダの中にある `*summary_all_features.csv` だけです。次のファイルは対象外です。
  - `外れ値除外_` で始まるファイル（01の出力のため）
  - 末端でないフォルダ（サブフォルダを持つフォルダ）に直接置かれたファイル
  - `.` で始まるフォルダ（`.ipynb_checkpoints` など）の中のファイル
- **出力ファイルを Excel で開いたまま実行しないでください。** 上書きできずにそのファイルだけ「エラー」になります（他のファイルの処理は続きます）。
- 必要な列がないCSVは「スキップ」になり、何も出力されません。
- 加工対象のパターンや実行するコードは、セル1の `TARGET_PATTERN`・`TARGET_NOTEBOOK` で変更できます。
- 詳しい説明は、プロジェクトフォルダ直下の `README.md` を見てください。

In [ ]:
# =====================================================================================
# セル1: ライブラリの準備と設定
# =====================================================================================
import os
import json
import fnmatch
import types
import traceback
import tkinter as tk
from tkinter import filedialog

# 一括処理で実行するコード（このノートブックと同じフォルダに置いておく）
TARGET_NOTEBOOK = '01_最高温度関連グラフ作成.ipynb'

# 加工対象ファイルのパターン
TARGET_PATTERN = '*summary_all_features.csv'

# 01コードが出力するCSV（上のパターンにも当てはまるため、加工対象から外す）
EXCLUDE_PREFIX = '外れ値除外_'

# グローバル変数の初期化
main_folder = ''
target_files = []

print("セル1の実行が完了しました。ライブラリと設定が読み込まれました。")

In [ ]:
# =====================================================================================
# セル2: メインフォルダの選択と加工対象ファイルの収集
# =====================================================================================
print("--- ステップ1: メインフォルダ選択 ---")
root = tk.Tk()
root.withdraw()
root.call('wm', 'attributes', '.', '-topmost', True) # ダイアログを最前面に

main_folder = filedialog.askdirectory(title="メインフォルダを選択してください")
target_files = []

if not main_folder:
    print("フォルダが選択されませんでした。処理を中断します。")
else:
    print(f"メインフォルダ: {main_folder}")
    print("--- ステップ2: 末端フォルダ内の加工対象ファイルを検索 ---")

    leaf_count = 0
    for dirpath, dirnames, filenames in os.walk(main_folder):
        # 「.」で始まるフォルダ（.ipynb_checkpoints など）は数えない。並び順も固定する
        dirnames[:] = sorted(d for d in dirnames if not d.startswith('.'))
        if dirnames:
            continue  # サブフォルダがあるフォルダは末端ではない

        leaf_count += 1
        for name in sorted(filenames):
            if fnmatch.fnmatch(name, TARGET_PATTERN) and not name.startswith(EXCLUDE_PREFIX):
                target_files.append(os.path.join(dirpath, name))

    print(f"末端フォルダ: {leaf_count}個 / 加工対象ファイル: {len(target_files)}件")
    for file_path in target_files:
        print(f"  {os.path.relpath(file_path, main_folder)}")

    if target_files:
        print("\n加工対象ファイルの検索が完了しました。次のセルへ進んでください。")
    else:
        print(f"\n加工対象ファイル（{TARGET_PATTERN}）が見つかりませんでした。")

In [ ]:
# =====================================================================================
# セル3: 01コードを加工対象ファイルごとに実行（出力は加工対象ファイルと同じフォルダ）
# =====================================================================================
target_notebook_path = os.path.join(os.getcwd(), TARGET_NOTEBOOK)

if not target_files:
    print("エラー: 加工対象ファイルがありません。セル2を先に実行してください。")
elif not os.path.exists(target_notebook_path):
    print(f"エラー: {TARGET_NOTEBOOK} が見つかりません。このノートブックと同じフォルダに置いてください。\n探した場所: {target_notebook_path}")
else:
    print(f"--- ステップ3: {TARGET_NOTEBOOK} の処理を一括実行 ---")

    # 01コードのセル（コード部分）を読み込む
    with open(target_notebook_path, encoding='utf-8') as f:
        code_cells = [''.join(cell['source']) for cell in json.load(f)['cells'] if cell['cell_type'] == 'code']

    class NoDialogRoot:
        """01コードのダイアログ用ウィンドウの代わり（何もしない）"""
        def withdraw(self): pass
        def call(self, *args): pass

    results = []
    for i, file_path in enumerate(target_files, 1):
        print(f"\n{'=' * 80}\n[{i}/{len(target_files)}] {os.path.relpath(file_path, main_folder)}\n{'=' * 80}")

        # ファイルごとに新しい名前空間で実行する（前のファイルのデータを持ち越さない）
        ns = {}
        try:
            exec(compile(code_cells[0], f"{TARGET_NOTEBOOK} セル1", 'exec'), ns)

            # 01コードのファイル選択ダイアログの代わりに、このファイルを選んだことにする
            ns['tk'] = types.SimpleNamespace(Tk=NoDialogRoot)
            ns['filedialog'] = types.SimpleNamespace(askopenfilenames=lambda **kwargs: (file_path,))

            exec(compile(code_cells[1], f"{TARGET_NOTEBOOK} セル2", 'exec'), ns)

            # 必要な列がなく01コードが読み込みをスキップしたファイルは、グラフ出力へ進まない
            if not ns['dataframes']:
                status = 'スキップ（必要な列なし）'
            else:
                for n, code in enumerate(code_cells[2:], 3):
                    exec(compile(code, f"{TARGET_NOTEBOOK} セル{n}", 'exec'), ns)
                status = '完了'
        except Exception as e:
            traceback.print_exc()
            status = f'エラー（{type(e).__name__}: {e}）'
        results.append((file_path, status))

    # 一括処理の結果まとめ
    print(f"\n{'=' * 80}\n■ 一括処理の結果\n{'=' * 80}")
    for file_path, status in results:
        print(f"  {status}: {os.path.relpath(file_path, main_folder)}")

    n_done = sum(status == '完了' for _, status in results)
    n_skip = sum(status.startswith('スキップ') for _, status in results)
    n_error = sum(status.startswith('エラー') for _, status in results)
    print(f"\n完了 {n_done}件 / スキップ {n_skip}件 / エラー {n_error}件")